In [ ]:
import sys, os
import random
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

#from src.data.data_collector import DataCollector
from src.models.model_trainer_rl_v2_2_buyhold import ModelTrainerRL, TradingEnvRL
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from src.models.backtester import PortfolioBacktester, PortfolioBacktesterRL
from src.utils.config_loader import load_config


config = load_config("config/config.yaml")
DEFAULT_SEED = 42

def set_global_seed(seed=DEFAULT_SEED):
    np.random.seed(seed)
    random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    return seed

## V2.2 PPO Agent

### Function

In [14]:
def run_ppo_trading_pipeline(
    stock_symbol,
    config,
    year='2022',
    save_path="models/",
    show_plot=True,
    seed=DEFAULT_SEED,
    n_eval_episodes=10,
):
    """
    Complete PPO trading pipeline: load data, train model, generate predictions, and backtest.
    
    Parameters:
    -----------
    stock_symbol : str
        Stock ticker symbol (e.g., 'CNP', 'MDU')
    config : dict
        Configuration dictionary loaded from config.yaml
    year : str, optional
        Year for data file selection (default: '2022')
    save_path : str, optional
        Directory to save/load models (default: 'models/')
    show_plot : bool, optional
        Whether to display the portfolio plot (default: True)
    seed : int, optional
        Fixed random seed for repeatable runs (default: 42)
    n_eval_episodes : int, optional
        Number of evaluation episodes for behavior metrics (default: 10)
    
    Returns:
    --------
    tuple : (portfolio, metrics, actions)
        - portfolio: VectorBT portfolio object
        - metrics: Dictionary of performance metrics
        - actions: Array of predicted actions
    """
    set_global_seed(seed)
    print(f"Using fixed seed: {seed}")
    
    # 1. Load Data
    try:
        data = pd.read_csv(f'data/processed/{stock_symbol}_processed_{year}.csv')
        if 'Date' in data.columns:
            data['Date'] = pd.to_datetime(data['Date'])
            data.set_index('Date', inplace=True)
        print(f"Data loaded successfully for {stock_symbol}.")
    except FileNotFoundError:
        print(f"Error: Data file not found for {stock_symbol}. Check path.")
        sys.exit()
    
    # 2. Split Train/Test
    split_idx = int(len(data) * 0.7)
    train_df = data.iloc[:split_idx]
    test_df = data.iloc[split_idx:]
    
    # 3. Training Phase
    print(f"Training PPO Agent for {stock_symbol}...")
    trainer = ModelTrainerRL(config['reinforcement_learning'])
    env_params = config['reinforcement_learning']['environment']
    
    env_train = TradingEnvRL(
        train_df,
        initial_balance=env_params.get('initial_balance', 10000),
        commission=env_params.get('commission', 0.001),
        lookback_window=env_params.get('lookback_window', 30),
        reward_func='profit'
    )
    
    result = trainer.train_ppo(env_train)
    trained_model = result['model']
    print(f"Training result keys: {list(result.keys())}")
    print(f"Trained model type: {type(trained_model).__name__}")
    if 'vec_env' in result:
        print(f"Vectorized env type: {type(result['vec_env']).__name__}")
    trainer.save_models(save_path)
    print("Training complete. Models saved.")
    
    # Extract PPO training diagnostics (if available from SB3 logger)
    ppo_diag = {}
    logger_values = getattr(getattr(trained_model, 'logger', None), 'name_to_value', {})
    for key in [
        'train/clip_fraction',
        'train/approx_kl',
        'train/entropy_loss',
        'train/policy_gradient_loss',
        'train/value_loss',
        'train/explained_variance',
    ]:
        val = logger_values.get(key, None) if isinstance(logger_values, dict) else None
        if val is not None:
            ppo_diag[key] = float(val)
    
    model_lr = None
    try:
        model_lr = float(trained_model.policy.optimizer.param_groups[0]['lr'])
    except Exception:
        model_lr = None
    
    # 4. Inference Phase
    print("Generating Agent Predictions on Test Data...")
    model = PPO.load(os.path.join(save_path, "ppo_model"))
    
    env_test = TradingEnvRL(
        test_df,
        initial_balance=env_params.get('initial_balance', 100000),
        commission=env_params.get('commission', 0.001),
        lookback_window=env_params.get('lookback_window', 30),
        reward_func='profit'
    )
    
    vec_env_test = DummyVecEnv([lambda: env_test])
    
    norm_path = os.path.join(save_path, "ppo_vecnormalize.pkl")
    if os.path.exists(norm_path):
        vec_env_test = VecNormalize.load(norm_path, vec_env_test)
        vec_env_test.training = False
        vec_env_test.norm_reward = False
    else:
        print("WARNING: Normalization stats not found. Model predictions may be garbage.")
    
    if hasattr(vec_env_test, "seed"):
        vec_env_test.seed(seed)
    obs = vec_env_test.reset()
    done = [False]
    actions = []
    
    while not done[0]:
        action, _ = model.predict(obs, deterministic=True)
        actions.append(float(action[0]))
        obs, _, done, _ = vec_env_test.step(action)
    
    print(f"Generated {len(actions)} actions.")
    
    # Evaluate behavior over multiple episodes for update-speed/stability/divergence metrics
    eval_rewards = []
    eval_lengths = []
    eval_returns = []
    for ep in range(max(1, int(n_eval_episodes))):
        set_global_seed(seed + ep)
        obs_ep, _ = env_test.reset(seed=seed + ep)
        done_ep = False
        ep_reward = 0.0
        ep_len = 0
        last_info = {}
        
        while not done_ep:
            action_ep, _ = model.predict(obs_ep, deterministic=True)
            obs_ep, reward_ep, terminated_ep, truncated_ep, info_ep = env_test.step(action_ep)
            done_ep = terminated_ep or truncated_ep
            ep_reward += float(reward_ep)
            ep_len += 1
            last_info = info_ep
        
        eval_rewards.append(ep_reward)
        eval_lengths.append(ep_len)
        eval_returns.append(float(last_info.get('return', 0.0)))
    
    # Build metric kwargs with proper parameter checking
    import inspect
    metric_kwargs = dict(
        episode_rewards=eval_rewards,
        episode_lengths=eval_lengths,
        episode_returns=eval_returns,
        success_threshold=0.0,
    )
    
    # Check which optional parameters the method accepts
    sig = inspect.signature(trainer.calculate_rl_performance_metrics)
    if 'random_state' in sig.parameters:
        metric_kwargs['random_state'] = seed
    if 'optimizer_learning_rate' in sig.parameters and model_lr is not None:
        metric_kwargs['optimizer_learning_rate'] = model_lr
    
    rl_metrics = trainer.calculate_rl_performance_metrics(**metric_kwargs)
    
    # Action saturation as an additional clipping proxy
    actions_arr = np.array(actions).flatten()
    eps = 1e-6
    action_low_boundary_pct = float(np.mean(actions_arr <= (0.0 + eps)) * 100.0) if len(actions_arr) > 0 else 0.0
    action_high_boundary_pct = float(np.mean(actions_arr >= (1.0 - eps)) * 100.0) if len(actions_arr) > 0 else 0.0
    
    # 5. Backtesting Phase
    print("Running Backtest...")
    backtester = PortfolioBacktesterRL(env_params)
    
    portfolio = backtester.run_backtest(
        price_data=test_df['close'],
        predicted_weights=np.array(actions).flatten(),
        lookback_window=env_params.get('lookback_window', 30)
    )
    
    comparison = backtester.compare_with_buy_and_hold_rl()
    metrics = backtester.get_performance_metrics()
    
    print(f"\n--- Strategy Performance for {stock_symbol} ---")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    
    print("\n--- Result Summary ---")
    print(f"Seed: {seed}")
    print(f"Actions generated: {len(actions)}")
    if isinstance(comparison, dict):
        print("Buy & Hold comparison:")
        for k, v in comparison.items():
            print(f"{k}: {v}")
    else:
        print(f"Buy & Hold comparison: {comparison}")
    
    print("\n--- Update Speed Metrics (New Incoming Data Adaptation) ---")
    print(f"Learning improvement: {rl_metrics.get('learning_improvement', 0.0):.6f}")
    print(f"Learning rate pct: {rl_metrics.get('learning_rate_pct', 0.0):.4f}%")
    print(f"Trend slope: {rl_metrics.get('trend_slope', 0.0):.6f}")
    
    print("\n--- Stability / Random Behavior Metrics ---")
    print(f"Mean episode reward: {rl_metrics.get('mean_episode_reward', 0.0):.6f}")
    print(f"Std episode reward: {rl_metrics.get('std_episode_reward', 0.0):.6f}")
    print(f"Coefficient of variation: {rl_metrics.get('coefficient_of_variation', 0.0):.4f}%")
    print(f"Variance ratio late/early: {rl_metrics.get('variance_reduction_pct', 0.0):.6f}")
    print(f"Success rate: {rl_metrics.get('success_rate_pct', 0.0):.2f}%")
    print(f"Win rate: {rl_metrics.get('win_rate_pct', 0.0):.2f}%")
    print(f"Optimizer learning rate: {rl_metrics.get('optimizer_learning_rate', 0.0):.6f}")
    
    print("\n--- Divergence / Failure-Case Features ---")
    print(f"Worst episode idx: {rl_metrics.get('worst_episode_idx', -1)}")
    print(f"Worst episode reward: {rl_metrics.get('worst_episode_reward', 0.0):.6f}")
    print(f"Worst episode return: {rl_metrics.get('worst_episode_return', 0.0):.6f}")
    print(f"Max drawdown pct: {rl_metrics.get('max_drawdown_pct', 0.0):.4f}%")
    negative_returns = int(np.sum(np.array(eval_returns) < 0.0))
    print(f"Negative-return episodes: {negative_returns}/{len(eval_returns)}")
    
    print("\n--- PPO Clipping Diagnostics ---")
    if len(ppo_diag) > 0:
        if 'train/clip_fraction' in ppo_diag:
            print(f"train/clip_fraction: {ppo_diag['train/clip_fraction']:.6f}")
        if 'train/approx_kl' in ppo_diag:
            print(f"train/approx_kl: {ppo_diag['train/approx_kl']:.6f}")
        if 'train/entropy_loss' in ppo_diag:
            print(f"train/entropy_loss: {ppo_diag['train/entropy_loss']:.6f}")
        if 'train/policy_gradient_loss' in ppo_diag:
            print(f"train/policy_gradient_loss: {ppo_diag['train/policy_gradient_loss']:.6f}")
        if 'train/value_loss' in ppo_diag:
            print(f"train/value_loss: {ppo_diag['train/value_loss']:.6f}")
        if 'train/explained_variance' in ppo_diag:
            print(f"train/explained_variance: {ppo_diag['train/explained_variance']:.6f}")
    else:
        print("PPO training clip diagnostics are not available from logger in this run.")
    print(f"Action boundary at 0.0 (%): {action_low_boundary_pct:.2f}%")
    print(f"Action boundary at 1.0 (%): {action_high_boundary_pct:.2f}%")
    
    if show_plot:
        portfolio.plot().show()
        # Print trade statistics
        trades = portfolio.trades.records_readable
        print(f"\n--- Trade Statistics for {stock_symbol} ---")
        print(f"Total number of trades: {len(trades)}")
        print("\nTrade Direction Counts:")
        print(trades['Direction'].value_counts())
        
        # Analyze trade outcomes
        if 'PnL' in trades.columns:
            profitable_trades = trades[trades['PnL'] > 0]
            loss_trades = trades[trades['PnL'] < 0]
            print(f"\n--- Trade Outcomes ---")
            print(f"Number of profitable exits: {len(profitable_trades)}")
            print(f"Number of loss exits (cut loss): {len(loss_trades)}")
            print(f"Win rate: {len(profitable_trades) / len(trades) * 100:.2f}%")
            print(f"Average profit per winning trade: ${profitable_trades['PnL'].mean():.2f}" if len(profitable_trades) > 0 else "No profitable trades")
            print(f"Average loss per losing trade: ${loss_trades['PnL'].mean():.2f}" if len(loss_trades) > 0 else "No losing trades")
    
    return portfolio, metrics, np.array(actions).flatten()


### Run

In [15]:
# Single stock quick demo run (reduced timesteps for fast output preview)
import copy

quick_config = copy.deepcopy(config)
quick_config['reinforcement_learning']['ppo']['total_timesteps'] = 200000

stock_symbol = "AAPL"
portfolio, metrics, actions = run_ppo_trading_pipeline(
    stock_symbol=stock_symbol,
    config=quick_config,
    show_plot=True,
    seed=DEFAULT_SEED,
    n_eval_episodes=10,
    save_path="models/quick_demo"
 )

C:\Users\BOOKLAPTOP\AppData\Local\Temp\ipykernel_31880\2817749670.py:44: FutureWarning:

In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`

INFO:src.models.model_trainer_rl_v2_2_buyhold:Training PPO...


Using fixed seed: 42
Data loaded successfully for AAPL.
Training PPO Agent for AAPL...
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 1083 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 869          |
|    iterations           | 2            |
|    time_elapsed         | 4            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0037352545 |
|    clip_fraction        | 0.0125       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | 0.0771       |
|    learning_rate        | 0.0002       |
|    loss                 | 0.11         |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00464     

INFO:src.models.model_trainer_rl_v2_2_buyhold:Training PPO for 200000 timesteps
C:\Users\BOOKLAPTOP\AppData\Local\Temp\ipykernel_31880\2817749670.py:129: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



Training result keys: ['model', 'vec_env']
Trained model type: PPO
Vectorized env type: VecNormalize
Saved model and normalization stats for models/quick_demo\ppo_vecnormalize.pkl
Training complete. Models saved.
Generating Agent Predictions on Test Data...
Generated 218 actions.


INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:src.models.model_trainer_rl_v2_2_buyhold:RL AGENT PERFORMANCE METRICS SUMMARY
INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:src.models.model_trainer_rl_v2_2_buyhold:Total Episodes: 10
INFO:src.models.model_trainer_rl_v2_2_buyhold:Mean Episode Reward: 0.00 ± 0.00
INFO:src.models.model_trainer_rl_v2_2_buyhold:Success Rate: 0.0%
INFO:src.models.model_trainer_rl_v2_2_buyhold:Learning Improvement: 0.00 (0.0%)
INFO:src.models.model_trainer_rl_v2_2_buyhold:Trend Slope: 0.0000
INFO:src.models.model_trainer_rl_v2_2_buyhold:Stability (CV): 0.00%
INFO:src.models.model_trainer_rl_v2_2_buyhold:Portfolio Sharpe Ratio: 0.000
INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:BacktesterRL:Preparing Backtest. Raw Prices: 279, Predictions: 218
INFO:BacktesterRL:Running

Running Backtest...

--- Strategy Performance for AAPL ---
Total Return (%): 103.5300
Annual Return (%): 228.6500
Sharpe Ratio: 3.7489
Sortino Ratio: 8.4616
Max Drawdown (%): -8.2900
Calmar Ratio: 27.5683
Win Rate (%): 88.2400
Total Trades: 17.0000
Final Value ($): 203526.7400

--- Result Summary ---
Seed: 42
Actions generated: 218
Buy & Hold comparison:                            RL Strategy  Buy & Hold
Date                                              
2025-01-10 00:00:00-05:00     0.000000   -0.000999
2025-01-13 00:00:00-05:00     0.000000   -0.011333
2025-01-14 00:00:00-05:00     0.000000   -0.016057
2025-01-15 00:00:00-05:00     0.000000    0.003303
2025-01-16 00:00:00-05:00     0.000000   -0.037230
...                                ...         ...
2025-11-14 00:00:00-05:00     1.054978    0.154185
2025-11-17 00:00:00-05:00     1.054978    0.133212
2025-11-18 00:00:00-05:00     1.054978    0.133127
2025-11-19 00:00:00-05:00     1.052925    0.137873
2025-11-20 00:00:00-05:00     1


--- Trade Statistics for AAPL ---
Total number of trades: 17

Trade Direction Counts:
Direction
Long    17
Name: count, dtype: int64

--- Trade Outcomes ---
Number of profitable exits: 15
Number of loss exits (cut loss): 2
Win rate: 88.24%
Average profit per winning trade: $7071.01
Average loss per losing trade: $-1269.17


In [4]:
# # portfolio.trades.records_readable
# # Print trade statistics
# trades = portfolio.trades.records_readable
# print(f"\n--- Trade Statistics for abc ---")
# print(f"Total number of trades: {len(trades)}")
# print("\nTrade Direction Counts:")
# print(trades['Direction'].value_counts())

# # Analyze trade outcomes
# if 'PnL' in trades.columns:
#         profitable_trades = trades[trades['PnL'] > 0]
#         loss_trades = trades[trades['PnL'] < 0]
        
#         print(f"\n--- Trade Outcomes ---")
#         print(f"Number of profitable exits: {len(profitable_trades)}")
#         print(f"Number of loss exits (cut loss): {len(loss_trades)}")
#         print(f"Win rate: {len(profitable_trades) / len(trades) * 100:.2f}%")
#         print(f"Average profit per winning trade: ${profitable_trades['PnL'].mean():.2f}" if len(profitable_trades) > 0 else "No profitable trades")
#         print(f"Average loss per losing trade: ${loss_trades['PnL'].mean():.2f}" if len(loss_trades) > 0 else "No losing trades")
    


In [10]:
# Multiple stocks in a loop
stocks = ["AAPL", "AMZN", "TSLA", "BAC","MDU", "CWCO", "NEE", "DUK"]
results = {}

for stock in stocks:
    print(f"\n{'='*60}")
    print(f"Processing {stock}")
    print(f"{'='*60}")
    portfolio, metrics, actions = run_ppo_trading_pipeline(
        stock_symbol=stock, 
        config=config,
        seed=DEFAULT_SEED,
        n_eval_episodes=10,
        show_plot=True  # Don't show plots in loop
    )
    results[stock] = {'portfolio': portfolio, 'metrics': metrics, 'actions': actions}

C:\Users\BOOKLAPTOP\AppData\Local\Temp\ipykernel_31880\3529423988.py:44: FutureWarning:

In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`

INFO:src.models.model_trainer_rl_v2_2_buyhold:Training PPO...



Processing AAPL
Using fixed seed: 42
Data loaded successfully for AAPL.
Training PPO Agent for AAPL...
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 1008 |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 864          |
|    iterations           | 2            |
|    time_elapsed         | 4            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0037352545 |
|    clip_fraction        | 0.0125       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | 0.0771       |
|    learning_rate        | 0.0002       |
|    loss                 | 0.11         |
|    n_updates            | 10           |
|    policy_gradient_los

INFO:src.models.model_trainer_rl_v2_2_buyhold:Training PPO for 200000 timesteps
C:\Users\BOOKLAPTOP\AppData\Local\Temp\ipykernel_31880\3529423988.py:123: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



Training result keys: ['model', 'vec_env']
Trained model type: PPO
Vectorized env type: VecNormalize
Saved model and normalization stats for models/ppo_vecnormalize.pkl
Training complete. Models saved.
Generating Agent Predictions on Test Data...
Generated 218 actions.


INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:src.models.model_trainer_rl_v2_2_buyhold:RL AGENT PERFORMANCE METRICS SUMMARY
INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:src.models.model_trainer_rl_v2_2_buyhold:Total Episodes: 10
INFO:src.models.model_trainer_rl_v2_2_buyhold:Mean Episode Reward: 0.00 ± 0.00
INFO:src.models.model_trainer_rl_v2_2_buyhold:Success Rate: 0.0%
INFO:src.models.model_trainer_rl_v2_2_buyhold:Learning Improvement: 0.00 (0.0%)
INFO:src.models.model_trainer_rl_v2_2_buyhold:Trend Slope: 0.0000
INFO:src.models.model_trainer_rl_v2_2_buyhold:Stability (CV): 0.00%
INFO:src.models.model_trainer_rl_v2_2_buyhold:Portfolio Sharpe Ratio: 0.000
INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:BacktesterRL:Preparing Backtest. Raw Prices: 279, Predictions: 218
INFO:BacktesterRL:Running

...

INFO:src.models.model_trainer_rl_v2_2_buyhold:Training PPO for 200000 timesteps
C:\Users\BOOKLAPTOP\AppData\Local\Temp\ipykernel_31880\3529423988.py:123: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



Training result keys: ['model', 'vec_env']
Trained model type: PPO
Vectorized env type: VecNormalize
Saved model and normalization stats for models/ppo_vecnormalize.pkl
Training complete. Models saved.
Generating Agent Predictions on Test Data...
Generated 218 actions.


INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:src.models.model_trainer_rl_v2_2_buyhold:RL AGENT PERFORMANCE METRICS SUMMARY
INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:src.models.model_trainer_rl_v2_2_buyhold:Total Episodes: 10
INFO:src.models.model_trainer_rl_v2_2_buyhold:Mean Episode Reward: 15.88 ± 0.00
INFO:src.models.model_trainer_rl_v2_2_buyhold:Success Rate: 100.0%
INFO:src.models.model_trainer_rl_v2_2_buyhold:Learning Improvement: 0.00 (0.0%)
INFO:src.models.model_trainer_rl_v2_2_buyhold:Trend Slope: 0.0000
INFO:src.models.model_trainer_rl_v2_2_buyhold:Stability (CV): 0.00%
INFO:src.models.model_trainer_rl_v2_2_buyhold:Portfolio Sharpe Ratio: 0.000
INFO:src.models.model_trainer_rl_v2_2_buyhold:============================================================
INFO:BacktesterRL:Preparing Backtest. Raw Prices: 279, Predictions: 218
INFO:BacktesterRL:Runn

Running Backtest...

--- Strategy Performance for DUK ---
Total Return (%): 38.4500
Annual Return (%): 72.4100
Sharpe Ratio: 3.0466
Sortino Ratio: 4.9697
Max Drawdown (%): -4.7300
Calmar Ratio: 15.3160
Win Rate (%): 85.7100
Total Trades: 14.0000
Final Value ($): 138447.6600

--- Result Summary ---
Seed: 42
Actions generated: 218
Buy & Hold comparison:                            RL Strategy  Buy & Hold
Date                                              
2025-01-10 00:00:00-05:00    -0.000999   -0.000999
2025-01-13 00:00:00-05:00     0.001549    0.001549
2025-01-14 00:00:00-05:00     0.008248    0.008248
2025-01-15 00:00:00-05:00     0.004474    0.004474
2025-01-16 00:00:00-05:00     0.029102    0.029102
...                                ...         ...
2025-11-14 00:00:00-05:00     0.386397    0.199784
2025-11-17 00:00:00-05:00     0.414304    0.223935
2025-11-18 00:00:00-05:00     0.398712    0.210442
2025-11-19 00:00:00-05:00     0.380974    0.195091
2025-11-20 00:00:00-05:00     0.38


--- Trade Statistics for DUK ---
Total number of trades: 14

Trade Direction Counts:
Direction
Long    14
Name: count, dtype: int64

--- Trade Outcomes ---
Number of profitable exits: 12
Number of loss exits (cut loss): 2
Win rate: 85.71%
Average profit per winning trade: $3416.61
Average loss per losing trade: $-1275.84
